# 03 — Luật kết hợp & Phân cụm

Mục tiêu:
- Tìm tổ hợp yếu tố dẫn đến nghỉ việc (Apriori)
- So sánh lift giữa Stay vs Leave
- Gợi ý chính sách HR
- Phân cụm nhân viên (KMeans, DBSCAN, HAC)
- Profiling cụm + HR Strategy mapping

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np

from src.data.loader import load_data, load_config
from src.data.cleaner import HRDataCleaner
from src.mining.association import get_attrition_rules, compare_lift_stay_vs_leave, suggest_hr_policies
from src.mining.clustering import run_kmeans, run_dbscan, run_hac, find_optimal_k, profile_clusters
from src.evaluation.metrics import build_clustering_comparison
from src.visualization.plots import (
    plot_cluster_profiles, plot_elbow_silhouette,
    plot_lift_comparison, plot_clustering_comparison,
)

config = load_config("configs/params.yaml")
df = load_data(config["data"]["raw_path"])
cleaner = HRDataCleaner(target_col="Attrition")
df_clean = cleaner.clean(df)

## 3.1 Luật kết hợp (Apriori)

In [ ]:
df_disc = cleaner.discretize_for_mining(df_clean)

rules = get_attrition_rules(
    df_disc,
    min_support=config["mining"]["min_support"],
    min_threshold=config["mining"]["min_threshold"],
)

if not rules.empty:
    display_cols = ["antecedents", "consequents", "support", "confidence", "lift"]
    rules[display_cols].head(15)

## 3.2 So sánh Lift: Stay vs Leave

> **Insight**: Các tổ hợp có Lift_Leave >> Lift_Stay là yếu tố chính gây nghỉ việc.

In [ ]:
lift_comp = compare_lift_stay_vs_leave(df_disc, min_support=config["mining"]["min_support"])
if not lift_comp.empty:
    plot_lift_comparison(lift_comp)
    lift_comp.head(15)

## 3.3 Gợi ý chính sách HR

> Dựa trên luật kết hợp, đề xuất chính sách cụ thể cho ban lãnh đạo.

In [ ]:
if not rules.empty:
    suggestions = suggest_hr_policies(rules, top_n=5)

## 3.4 Phân cụm — Tìm K tối ưu

In [ ]:
from sklearn.preprocessing import StandardScaler

cluster_features = config["mining"]["cluster_features"]
cluster_features = [f for f in cluster_features if f in df_clean.columns]

X_cluster = StandardScaler().fit_transform(df_clean[cluster_features])
optimal = find_optimal_k(X_cluster)
plot_elbow_silhouette(optimal)

## 3.5 So sánh phương pháp Clustering

In [ ]:
# K-Means
df_km, _, sil_km = run_kmeans(df_clean, cluster_features, n_clusters=config["mining"]["n_clusters"])

# DBSCAN
df_db, _, sil_db = run_dbscan(df_clean, cluster_features, eps=config["mining"]["dbscan_eps"])
n_db = len(set(df_db["Cluster"])) - (1 if -1 in df_db["Cluster"].values else 0)

# HAC
df_hac, _, sil_hac = run_hac(df_clean, cluster_features, n_clusters=config["mining"]["n_clusters"])

# So sánh
comp_data = {
    "K-Means": {"silhouette": sil_km, "n_clusters": config["mining"]["n_clusters"]},
    "DBSCAN": {"silhouette": sil_db, "n_clusters": n_db},
    "HAC (Ward)": {"silhouette": sil_hac, "n_clusters": config["mining"]["n_clusters"]},
}
build_clustering_comparison(comp_data)
plot_clustering_comparison(comp_data)

## 3.6 Profiling cụm + HR Strategy

> **Cluster → HR Strategy mapping**: mỗi cụm có đặc điểm riêng và chiến lược can thiệp khác nhau.

In [ ]:
profile = profile_clusters(df_km, cluster_features)
plot_cluster_profiles(profile[cluster_features])
profile